# LGAD Module simulator

## 1.- Goals of this notebook

* Understand all the aspects of the semi-empirical LGAD response module
* Visualize a typical signal
* Produce datasets containing the response of the LGAD + ETROC

## 2.- Imports needed by the program

This program makes use of 4 modules:

* **LGADSimulator**: This module implements the LGAD semi-empirical model. 
* **numpy**: This is the standard python package for numerical calculations.
* **matplotlib**: This is the standard python package for plotting data.
* **pandas**: This is one of the most popular packages to handle datasets.

In [ ]:
from LGADModuleSimulator.src.LGADSimulator import LGADSimulator
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

('dataset3.h5', <http.client.HTTPMessage at 0x7a10144b6350>)

## 3.- Have a look at the semi-empirical LGAD+ETROC simulator model

In the workbook of this exercise you will find a brief description of the model used to generate the LGAD+ETROC response, however it is important that you have a look at the code and understand how it works in detail. 

### Model parameters

* **Thickness**: This is just the thickness of the active area of the sensor. It is given in cm. 
* **Radius**: This is the distance at which this sensor is located with respect to the beam line. Take into account that in the case of CMS these sensors will be installed in the Endcap, in such a way that this distance determines the amount of radition that the sensor will be exposed to. It is given in cm.
* **intLumi**: This is the integrated luminosity. Although in this exercise, radiation effects will not be considered, the degradation of the sensors due to radiation damage can be taken into account in this model. Given in inverse femtobarns.
* **taur**: This is the time constant regulating the rise time of the LGAD+ETROC signal shape. It is given in nanoseconds. 
* **taud**: This is the time constant regulating the down time of the LGAD+ETROC signal shape. It is given in nanoseconds.
* **clock**: This is the frecuency of the clock of the system. It is measured in MHz.
* **treshold**: The treshold that will trigger the acquisition of an event. It is measured in fC. 
* **noiseLevel**: The RMS of the electronic noise in the signal. It is measure in fC. 
* **sigmaTDC**: This is the time resolution associated to the TDC conversor.  

### Input to generate an event

Once a model is setup events can be generated. The events depend on 4 parameters: 

* **id**: The type of particle according to the PDG.
* **p**: The momentum of the parcicle. It is given in GeV.
* **angle**: The incidence angle. It is given in radian.
* **t**: The actual entrance time to the sensor. It is given in nanoseconds.

### Output of the event generator

For a given set of input variables the model generates two values:

* **TOA**: The Time Of Arrival of the particle. It is given in ns.
* **TOT**: The Time Over Threshold of the signal. It is given in ns.

## 4.- Some questions before your continue (Please edit this cell and add your answers)

Q. How is the energy deposited in the sensor modeled? 

A.

Q. What is the Gain of an LGAD sensor? 

A.

Q. How is the gain behaving with respect to the radiation dose received?

A.

Q. Why is the energy deposited relevant in this system?

A.

Q. What is the parameter "signalTMax"?

A.

Q. What is the parameter "signalMax"?

A. 


## 5.- Setting up two models with different parameters

In this section you will have to create two different models with the following suggested parameters:

### Model 1

thickness = 0.3

radius = 100.0

intLumi = 0.0

taur = 2.0

taud = 3.0

clock = 3.0

threshold = 0.9525

noiseLevel = 0.2750

sigmaTDC = 0.010

### Model 2

thickness = 0.3

radius = 100.0

intLumi = 0.0

taur = 2.0

taud = 3.0

clock = 3.0

threshold = 0.9525

noiseLevel = 0.2750

sigmaTDC = 0.010


In [ ]:

model1 = LGADSimulator(thickness=0.3, radius=1.0, 
                        intLumi = 0.0, taur = 2.0,
                        taud=3.0, clock=3.0,
                        threshold=0.9525, noiseLevel=0.2750, sigmaTDC=0.010)

model2 = LGADSimulator(thickness=0.3, radius=1.0, 
                        intLumi = 0.0, taur = 5.0,
                        taud=3.0, clock=3.0,
                        threshold=0.9525, noiseLevel=0.2750, sigmaTDC=0.010)

## 6.- Draw one event for each model

Generate one event with each of the generators and draw the signal shapes and the tresholds. 

In [ ]:
fig, ax = plt.subplots()
model1.drawEvent(ax, 121, 10, 0, 0, 'b')
model2.drawEvent(ax, 121, 10, 0, 0, 'r')

## 5.- Calculation of the POCA

Nothing has to be done in this function, but please take your time to see the implementation and understand the geometrical concept behind.

In [ ]:
import numpy as np

def getPoint(r1, r2, v1, v2):

    #Calculation of the closest point of approach
    cross_st = np.cross(v1, v2)
    cross_stnorm = np.linalg.norm(cross_st)
    vts = np.dot(v1, v2)
    if cross_stnorm < 1.0e-6 or vts < 1.0e-6:
        return False, [0, 0, 0]
    cross_sst = np.cross(v1, cross_st)
    DeltaR = r1 - r2
    xpoca2 = r2 - v2 * np.dot(DeltaR, cross_sst)/cross_stnorm**2
    xpoca1 = r1 + v1 * np.dot((xpoca2-r1), v1)/vts
    v = 0.5 * (xpoca1 + xpoca2)
    return True, v

## 6.- Looping through the dataset

This part of the code loops through the dataset and stores the POCA points in arrays. Please take your time to understand the selection criteria applied. 


In [ ]:
ax = []
ay = []
az = []

# loop through the rows using iterrows()
for index, row in dataset.iterrows():
    #if index > 100:
    #    break
    r1 = np.asarray([row['x1'], row['y1'], row['z1']])
    r2 = np.asarray([row['x2'], row['y2'], row['z2']])
    v1 = np.asarray([row['vx1'], row['vy1'], row['vz1']])
    v2 = np.asarray([row['vx2'], row['vy2'], row['vz2']])
    dtx = row['dthetax']
    dty = row['dthetay']

    valid = False
    ###Apply here a simple angular selection
    if abs(dtx) > binInfo['threshold1'] or abs(dty) > binInfo['threshold2']:
        valid, v = getPoint(r1, r2, v1, v2)
        if not valid:
            continue
        if v[0] < binInfo['limitX'][0] or v[0] > binInfo['limitX'][1]:
            continue
        if v[1] < binInfo['limitY'][0] or v[1] > binInfo['limitY'][1]:
            continue
        if v[2] < binInfo['limitZ'][0] or v[2] > binInfo['limitZ'][1]:
            continue
        ax.append(v[0])
        ay.append(v[1])
        az.append(v[2])
    else:
        continue
    x = np.asarray(ax)
    y = np.asarray(ay)
    z = np.asarray(az)

## 7.- Create all the plots 

This piece of code creates all the plots. Later on you will have to modify and add more plots. 

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(16, 5))
ax[0].set_title('Frontal view XY')
ax[1].set_title('Side view XZ')
ax[2].set_title('Side view YZ')
ax[0].set_xlabel('X [cm]')
ax[0].set_ylabel('Y [cm]')
ax[1].set_xlabel('X [cm]')
ax[1].set_ylabel('Z [cm]')
ax[2].set_xlabel('Y [cm]')
ax[2].set_ylabel('Z [cm]')

ax[0].hist2d(x, y, bins=(binInfo['xynbinx'], binInfo['xynbiny']), cmap=plt.cm.jet)
ax[1].hist2d(x, z, bins=(binInfo['xznbinx'], binInfo['xznbinz']), cmap=plt.cm.jet)
ax[2].hist2d(y, z, bins=(binInfo['yznbiny'], binInfo['yznbinz']), cmap=plt.cm.jet)


## 8.- Exercises and questions

##### Q1. Start with the first dataset. Tune the parameters until you see something meaninful in the plots. Take into account that looping through the dataset takes some time, so try to limit the number of steps but thinking about the geometry described in the slides.

##### Q2. Take a look at the three projections XY, XZ, and YZ. Is the resolution the same in all of them? Which one is better? Try to explain why.

##### Q3. Implement two additional plots showing the 1D distribution of the x and y angular distributions.

##### Q4. Use now the second dataset with the parameters you found before. What do you see? Do you have an hypothesis about what's going on with this dataset?

##### Q5. Can you improve the previous images by tuning the angular cuts?

##### Q6. Have a look at the 1D angular distributions. What do you see?

##### Q7. Use now the third dataset with the parameters you found for the first dataset. What do you see? Do you have an hypothesis about what's going on with this dataset?

##### Q8. Can you improve the previous image by tuning the angular cuts?

##### Q9. Have a look at the 1D angular distributions. What do you see?